"""
# NIFTY 7-High/Low Breakout Options Backtest

## Strategy Description:
- **Signal Generation:** 
    - Signals are generated using NIFTY spot index hourly candles.
    - **Bullish Entry:** When Close > highest Close of previous 7 bars AND Close > EMA200.
    - **Bearish Entry:** When Close < lowest Close of previous 7 bars AND Close < EMA200.

- **Trade Execution:**
    - **Bullish Signal:** Short ATM Put Option (PE) of the nearest weekly expiry.
    - **Bearish Signal:** Short ATM Call Option (CE) of the nearest weekly expiry.
    - ATM Strike = Closest strike to spot at entry time (rounded to nearest 50/100 as available).
    - Option prices are taken at the **last traded price at or before the entry/exit signal time** (never after).
    - All trades are closed on the exit signal, or **force-exited at the last available tick before expiry** if signal doesn't arrive before expiry.

- **Position Sizing:**
    - 1 lot per trade 

- **PnL Calculation:**
    - For each trade: **PnL = Entry Price - Exit Price** (short option logic).
    - Missing prices at entry/exit are tracked and reported.

- **Backtest Output:**
    - Records entry/exit times, strike, option type, expiry, prices, PnL, and relevant statistics.
    - Statistics include total PnL, win rate, average PnL, and count of trades with missing option data.


"""


In [ ]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time


In [ ]:
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/_NIFTY_IDX__202507041318.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

# Save and reload to Parquet (optional)
try:

    print("spot_df loaded successfully from parquet.")
except Exception as e:
    print(f"Error saving/loading spot_df: {e}")


In [ ]:
# load spot data

# --- 2. Ensure Datetime is datetime and sort for time-based ops ---
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# --- 3. Set Datetime as index for resampling ---
spot_df = spot_df.set_index('Datetime')

# --- 4. Filter to regular NIFTY trading hours (avoid pre/post-market ticks) ---
spot_df = spot_df.between_time('09:15:00', '15:15:00')

# --- 5. Resample to 1-min OHLC bars ---
spot_1min = spot_df.resample(
    '1min',
    origin='start_day',
    label='left',
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_1min = spot_1min.reset_index()

# --- 7. Calculate 200-period EMA on Close for trend filter ---
spot_1min['EMA_200'] = spot_1min['Close'].ewm(span=200, adjust=False).mean()

# --- 9. Preview final 1-min OHLC + EMA DataFrame ---
print(spot_1min.tail())


In [ ]:
#GENERATE SIGNALS CORRECT
# --- PREP ---
spot_1min = spot_1min.sort_values('Datetime').reset_index(drop=True)

# Pre-calculate rolling indicators
spot_1min['7bar_High'] = spot_1min['High'].shift(1).rolling(window=7).max()
spot_1min['7bar_Low'] = spot_1min['Low'].shift(1).rolling(window=7).min()
spot_1min['7bar_Close_High'] = spot_1min['Close'].shift(1).rolling(window=7).max()
spot_1min['7bar_Close_Low'] = spot_1min['Close'].shift(1).rolling(window=7).min()

signals = []
in_position = None   # None, "Bullish", "Bearish"

for i in range(len(spot_1min)-1):
    row = spot_1min.iloc[i]
    next_row = spot_1min.iloc[i+1]
    dt = next_row['Datetime']
    ema = row['EMA_200']
    close = row['Close']
    low = row['Low']
    high = row['High']

    # --- When flat, decide on first entry side using EMA 200 ---
    if in_position is None:
        if close > ema:
            # Only consider Bullish_Entry, using EMA filter
            entry_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
            if entry_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bullish_Entry', 'Direction': 'Bullish'})
                in_position = "Bullish"
        elif close < ema:
            # Only consider Bearish_Entry, using EMA filter
            entry_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
            if entry_cond:
                signals.append({'Datetime': dt, 'Signal': 'Bearish_Entry', 'Direction': 'Bearish'})
                in_position = "Bearish"
        # If close == ema or no entry, remain flat

    # --- When already in a trade (re-entries & exits ignore EMA filter) ---
    elif in_position == "Bullish":
        # Bullish Reentry/Exit (IGNORE EMA)
        entry_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
        exit_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
        if entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bullish_Reentry', 'Direction': 'Bullish'})
        elif exit_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bullish_Exit', 'Direction': 'Bullish'})
            in_position = None

    elif in_position == "Bearish":
        # Bearish Reentry/Exit (IGNORE EMA)
        entry_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
        exit_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
        if entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bearish_Reentry', 'Direction': 'Bearish'})
        elif exit_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bearish_Exit', 'Direction': 'Bearish'})
            in_position = None

# Output all signals as a DataFrame
signals_df = pd.DataFrame(signals)[['Datetime', 'Signal', 'Direction']]
print(signals_df.head(20))
print(f"Total signals generated: {len(signals_df)}")


In [ ]:
#MAKE TRADELOGS


# Ensure correct types/sorting
spot_1min['Datetime'] = pd.to_datetime(spot_1min['Datetime'])
signals_df['Datetime'] = pd.to_datetime(signals_df['Datetime'])
spot_1min = spot_1min.sort_values('Datetime').reset_index(drop=True)
signals_df = signals_df.sort_values('Datetime').reset_index(drop=True)

# Merge to get prices for signal times
signals_df = pd.merge(
    signals_df, 
    spot_1min[['Datetime', 'Open', 'Close']],
    how='left', 
    on='Datetime'
)

trade_log = []
open_trades = []

for idx, row in signals_df.iterrows():
    signal = row['Signal']
    dt = row['Datetime']
    price = row['Open']  # always use open price of the signal candle for entries/exits
    direction = row['Direction']
    
    # ENTRY/REENTRY: add new open trade
    if "Entry" in signal or "Reentry" in signal:
        open_trades.append({
            'Entry Time': dt,
            'Entry Price': price,
            'Side': direction,
            'Signal Type': signal,
            'Signal Index': idx
        })
    # EXIT: close all open trades on this side
    elif "Exit" in signal:
        to_close = [t for t in open_trades if t['Side'] == direction]
        for trade in to_close:
            trade['Exit Time'] = dt
            trade['Exit Price'] = price
            trade['Exit Signal Index'] = idx
            # PnL: Long = Exit - Entry, Short = Entry - Exit
            if direction == 'Bullish':
                trade['PnL'] = trade['Exit Price'] - trade['Entry Price']
            else:
                trade['PnL'] = trade['Entry Price'] - trade['Exit Price']
            trade_log.append(trade)
        open_trades = [t for t in open_trades if t['Side'] != direction]

# If any positions still open at the end, close at last available price
if open_trades:
    final_price = spot_1min.iloc[-1]['Open']
    final_time = spot_1min.iloc[-1]['Datetime']
    for trade in open_trades:
        trade['Exit Time'] = final_time
        trade['Exit Price'] = final_price
        if trade['Side'] == 'Bullish':
            trade['PnL'] = final_price - trade['Entry Price']
        else:
            trade['PnL'] = trade['Entry Price'] - final_price
        trade['Forced Exit'] = True
        trade_log.append(trade)

trade_log_df = pd.DataFrame(trade_log)

# Select clean columns for output
trade_log_df = trade_log_df[
    ['Entry Time', 'Exit Time', 'Side', 'Entry Price', 'Exit Price', 'Signal Type', 'PnL']
]

# Save for further use
#trade_log_df.to_parquet("7hl_tradelog_index.parquet", index=False)
print(trade_log_df.tail(20))
print(f"Total trades: {len(trade_log_df)}")
print(f"Win rate: {(trade_log_df['PnL'] > 0).mean() * 100:.2f}% | Avg PnL: {trade_log_df['PnL'].mean():.2f}")


In [ ]:
# Drop duplicate rows, keeping the first occurrence
trade_log_df = trade_log_df.drop_duplicates()

# If you want to reset the index after dropping duplicates
trade_log_df = trade_log_df.reset_index(drop=True)


In [ ]:
trade_log_df.to_excel("trade_log.xlsx", index=False)
trade_log_df.tail(20)

In [ ]:
#LOAD AND SAVE OPTION DATA SEPARATELY
years_to_load = [2021, 2022, 2023, 2024, 2025]

outdir = "./options_data_yearwise"
os.makedirs(outdir, exist_ok=True)

for year in years_to_load:
    print(f"\n--- Loading option data for year: {year} ---")

    # === NEAREST EXPIRY ===
    options_files = glob.glob(f"/home/newberry3/main/Data/NIFTY/NIFTY_{year}*.pkl")
    if not options_files:
        print(f"No .pkl files found in /main/Data/NIFTY for year {year}")
        continue

    all_cols = set()
    dfs = []
    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        all_cols.update(df.columns)
    all_cols = sorted(list(all_cols))

    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        for col in all_cols:
            if col not in df.columns:
                df[col] = pd.NA
        df = df[all_cols]
        dfs.append(df)
    options_df = pd.concat(dfs, ignore_index=True)
    for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
        if col in options_df.columns:
            options_df[col] = options_df[col].astype(str)
    options_df['DateTime'] = pd.to_datetime(options_df['Date'] + " " + options_df['Time'], errors='coerce')
    options_df = options_df.dropna(subset=['DateTime'])
    cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date','Time']
    options_df = options_df.drop(columns=[col for col in cols_to_drop if col in options_df.columns])
    cols = ['DateTime'] + [col for col in options_df.columns if col != 'DateTime']
    options_df = options_df[cols]
    options_df['StrikePrice'] = options_df['StrikePrice'].astype(float)
    options_df['Type'] = options_df['Type'].str.strip().str.upper()
    options_df['ExpiryDate'] = pd.to_datetime(options_df['ExpiryDate']).dt.normalize()

    # --- Save nearest expiry for this year ---
    nearest_path = os.path.join(outdir, f"options_nearest_{year}.parquet")
    options_df.to_parquet(nearest_path, index=False)
    print(f"✅ Saved NEAREST expiry options for {year} to {nearest_path}")

    # === NEXT EXPIRY ===
    options_files_2 = glob.glob(f"/home/newberry3/main/Data/2ndweeknext_Expiry/NIFTY_{year}*.pkl")
    if not options_files_2:
        print(f"No .pkl files found in /2ndweeknext_Expiry for year {year}")
        continue

    all_cols_2 = set()
    for file in options_files_2:
        df_2 = pickle.load(open(file, "rb"))
        df_2 = df_2.drop(columns=[col for col in ['OI', 'Volume'] if col in df_2.columns], errors='ignore')
        all_cols_2.update(df_2.columns)
    all_cols_2 = sorted(list(all_cols_2))

    dfs_2 = []
    for file in options_files_2:
        df_2 = pickle.load(open(file, "rb"))
        df_2 = df_2.drop(columns=[col for col in ['OI', 'Volume'] if col in df_2.columns], errors='ignore')
        for col in all_cols_2:
            if col not in df_2.columns:
                df_2[col] = pd.NA
        df_2 = df_2[all_cols_2]
        # Drop NA in key columns
        key_cols = ['StrikePrice', 'ExpiryDate', 'Date', 'Time', 'Type']
        df_2 = df_2.dropna(subset=[col for col in key_cols if col in df_2.columns])
        # Parse DateTime
        dt_strings = df_2['Date'].astype(str) + ' ' + df_2['Time'].astype(str)
        df_2['DateTime'] = pd.to_datetime(dt_strings, errors='coerce')
        df_2 = df_2.dropna(subset=['DateTime'])
        dfs_2.append(df_2)
    options_df_2 = pd.concat(dfs_2, ignore_index=True)
    for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
        if col in options_df_2.columns:
            options_df_2[col] = options_df_2[col].astype(str)
    options_df_2['StrikePrice'] = options_df_2['StrikePrice'].astype(float)
    options_df_2['Type'] = options_df_2['Type'].str.strip().str.upper()
    options_df_2['ExpiryDate'] = pd.to_datetime(options_df_2['ExpiryDate']).dt.normalize()
    cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date','Time']
    options_df_2 = options_df_2.drop(columns=[col for col in cols_to_drop if col in options_df_2.columns])
    cols = ['DateTime'] + [col for col in options_df_2.columns if col != 'DateTime']
    options_df_2 = options_df_2[cols]

    # --- Save next expiry for this year ---
    next_path = os.path.join(outdir, f"options_next_{year}.parquet")
    options_df_2.to_parquet(next_path, index=False)
    print(f"✅ Saved NEXT expiry options for {year} to {next_path}")

print(f"\nAll options data saved to {outdir}")


In [ ]:
#SAVE IN 3 MONTH POCKETS

data_dir = "./options_data_yearwise"
outdir = "./options_data_quarterly"
os.makedirs(outdir, exist_ok=True)

years_to_load = [2021, 2022, 2023, 2024, 2025]

# Define pockets: 3 months from June 2021 to June 2025
start_date = pd.Timestamp("2021-06-01")
end_date = pd.Timestamp("2025-06-30")
quarters = pd.date_range(start=start_date, end=end_date, freq='3MS')
pockets = []
for i in range(len(quarters)):
    pocket_start = quarters[i]
    pocket_end = pocket_start + pd.DateOffset(months=3) - pd.DateOffset(days=1)
    if pocket_end > end_date:
        pocket_end = end_date
    pockets.append((pocket_start, pocket_end))

def save_quarterly_pockets(options_df, label, year, outdir):
    for pocket_start, pocket_end in pockets:
        mask = (options_df['DateTime'] >= pocket_start) & (options_df['DateTime'] <= pocket_end)
        df_pocket = options_df[mask]
        if not df_pocket.empty:
            fname = f"{label}_{year}_{pocket_start.strftime('%Y%m%d')}_{pocket_end.strftime('%Y%m%d')}.parquet"
            df_pocket.to_parquet(os.path.join(outdir, fname), index=False)
            print(f"Saved {fname}: {len(df_pocket)} rows")

for year in years_to_load:
    nearest_path = os.path.join(data_dir, f"options_nearest_{year}.parquet")
    next_path = os.path.join(data_dir, f"options_next_{year}.parquet")

    # Load each DataFrame
    try:
        options_df = pd.read_parquet(nearest_path)
        options_df_2 = pd.read_parquet(next_path)
        print(f"Loaded: {nearest_path} and {next_path}")
    except Exception as e:
        print(f"Skipping year {year} due to load error: {e}")
        continue

    # --- Save 3M pockets for nearest expiry
    save_quarterly_pockets(options_df, "options_nearest", year, outdir)
    # --- Save 3M pockets for next expiry
    save_quarterly_pockets(options_df_2, "options_next", year, outdir)

    # --- Merge and save merged pockets
    merged = pd.concat([options_df, options_df_2], ignore_index=True)
    merged = merged.drop_duplicates(
        subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
    ).reset_index(drop=True)
    merged = merged.sort_values(['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'])

    save_quarterly_pockets(merged, "options_merged", year, outdir)

print("\n✅ All data saved as 3-month pockets in", outdir)


In [ ]:
# data_dir = "./options_data_yearwise"

# # Dictionaries to store DataFrames by year
# nearest_dict = {}
# next_dict = {}
# merged_dict = {}

# for year in years_to_load:
#     nearest_path = os.path.join(data_dir, f"options_nearest_{year}.parquet")
#     next_path = os.path.join(data_dir, f"options_next_{year}.parquet")

#     # Load each DataFrame and assign to dynamic variable names
#     try:
#         options_df = pd.read_parquet(nearest_path)
#         options_df_2 = pd.read_parquet(next_path)
#         print(f"Loaded: {nearest_path} and {next_path}")
#     except Exception as e:
#         print(f"Skipping year {year} due to load error: {e}")
#         continue

#     # Store individually
#     nearest_dict[year] = options_df
#     next_dict[year] = options_df_2

#     # Merge as per your logic
#     merged = pd.concat([options_df, options_df_2], ignore_index=True)
#     merged = merged.drop_duplicates(
#         subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
#     ).reset_index(drop=True)
#     merged = merged.sort_values(['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'])

#     # Store merged DataFrame
#     merged_dict[year] = merged

#     # Optionally: Save merged for each year
#     merged_path = os.path.join(data_dir, f"options_df_merged_{year}.parquet")
#     merged.to_parquet(merged_path, index=False)
#     print(f"✅ Saved merged DataFrame for {year} to {merged_path}")

# # Now you have:
# # nearest_dict[2024], next_dict[2024], merged_dict[2024], etc.
# # Or if you want variables, you can also:
# for year in years_to_load:
#     globals()[f"options_df_{year}"] = nearest_dict.get(year)
#     globals()[f"options_df_2_{year}"] = next_dict.get(year)
#     globals()[f"options_df_merged_{year}"] = merged_dict.get(year)

In [ ]:
# BS/IV functions

# --- Black-Scholes Option Price ---
def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None
    
# --- Implied Volatility from Option Price ---
def implied_volatility(option_price, S, K, T, r, option_type):
    """
    Uses Brent's method to find implied volatility from the market price.
    """
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None
    
# --- Black-Scholes Greeks ---
def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None

    # Use synthetic future price
    F = S * np.exp(r * T)

    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
    vega = F * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Delta': round(delta, 5),
        'Gamma': round(gamma, 5),
        'Vega': round(vega, 5),
        'Theta': round(theta, 5),
        'Rho': round(rho, 5)
    }

# --- Time to Expiry (Fractional) ---
def calculate_time_to_expiry(manual_datetime_str, expiry_date_str):
    now = datetime.strptime(manual_datetime_str, "%Y-%m-%d %H:%M:%S")
    expiry_date = datetime.strptime(expiry_date_str, "%d-%m-%y").date()

    market_open = time(9, 15)
    market_close = time(15, 30)
    today = now.date()
    days_left = (expiry_date - today).days

    if days_left <= 0:
        days_left += 1

    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)

    if current_minutes_since_open < 0:
        T = round(days_left / 365, 6)
    elif current_minutes_since_open >= total_trading_minutes:
        T = round(max(0, (days_left - 1) / 365), 6)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = round((days_left - fraction_of_day_passed) / 365, 6)

    print(f"\nManual Time Entered: {now}")
    print(f"Expiry Date: {expiry_date}")
    print(f"Days to Expiry (with fraction): {days_left - (fraction_of_day_passed if 0 <= current_minutes_since_open < total_trading_minutes else 0):.6f}")
    print(f"Time to Expiry in Years (T): {T}")

    return T

In [ ]:
def find_best_strike_for_signal(row, options_df, options_df_2, r=0.066):
   
 # --- Extract info ---
    entry_time = pd.to_datetime(row['Entry Time'])
    spot = row['Entry Price']
    side = row['Side']
    opt_type = 'PE' if side == 'Bullish' else 'CE'
    target_delta = -0.4 if side == 'Bullish' else 0.4

    # --- Pick correct DataFrame and Expiry ---
    weekday = entry_time.weekday()  # 0=Monday, ..., 4=Friday
    if weekday == 4:
        df = options_df
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEAREST (Friday)'
    else:
        df = options_df_2
        expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
        expiry_source = 'NEXT (Mon-Thu)'

    # Ensure correct types
    df = df.copy()
    df['StrikePrice'] = df['StrikePrice'].astype(float)
    df['ExpiryDate'] = pd.to_datetime(df['ExpiryDate'])
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['Type'] = df['Type'].astype(str)

    # Select expiry
    if expiry_list.empty:
        print(f"[DEBUG] No expiry found for {entry_time} ({side}), source={expiry_source}")
        return None
    expiry = pd.to_datetime(expiry_list.iloc[0])

    # Build time window (here, zero minutes—exact match only as in your code)
    time_window = pd.Timedelta(minutes=1)
    time_diff = (df['DateTime'] - entry_time).abs()

    # Filtering options: entry datetime, expiry, type, time within window
    mask = (
        (df['DateTime'].dt.floor('min') == entry_time.floor('min')) &  # For minute-level exact match
        (df['ExpiryDate'] == expiry) &
        (df['Type'] == opt_type) &
        (time_diff <= time_window)
    )
    df_opts = df[mask].copy()
    print(f"[DEBUG] Entry: {entry_time}, Expiry used: {expiry.date()}, "
          f"Type: {opt_type}, Rows in window (±0min): {len(df_opts)}, Source: {expiry_source}")

    if df_opts.empty:
        print(f"[DEBUG] NO OPTION DATA: {entry_time}, Expiry: {expiry.date()}, "
              f"Type: {opt_type}, Data source: {expiry_source}")
        return None

    best_row = None
    best_delta_diff = np.inf

    for idx, opt in df_opts.iterrows():
        K = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"), expiry.strftime("%d-%m-%y"))
        option_type_bs = 'put' if opt_type == 'PE' else 'call'

        # --- IV calculation ---
        iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
        if iv is None or iv <= 0:
            print(f"  [DEBUG] IV fail at strike {K} (price={option_price:.2f}), T={T:.6f}")
            continue

        greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
        if greeks is None:
            print(f"  [DEBUG] Greeks fail at strike {K} (IV={iv:.4f})")
            continue

        delta = greeks['Delta']
        delta_diff = abs(delta - target_delta)
        print(f"  [DEBUG] Strike {K}, Delta={delta:.5f}, IV={iv:.4f}, Δ={delta_diff:.5f}")

        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': delta,
                'Option Type': opt_type,
                'Option Price': option_price
            }
    if best_row is None:
        print(f"[DEBUG] No strike within delta range for {entry_time} {side}, expiry={expiry.date()}")
    return best_row

# # Usage
# results = []
# for idx, row in trade_log_df.iterrows():
#     res = find_best_strike_for_signal(row, options_df, options_df_2)
#     results.append(res)

# strikes_df = pd.DataFrame([x for x in results if x is not None])
# print(strikes_df)


In [ ]:
# #FINDING STRIKES AND ENTRY PRICES YEARWISE

# all_results = []

# # If you used globals() to define options_df_2021, etc.
# years_in_trades = trade_log_df['Entry Time'].dt.year.unique()

# for year in sorted(years_in_trades):
#     print(f"Processing trades for year: {year}")
    
#     # Subset trade log for this year only
#     trades_this_year = trade_log_df[trade_log_df['Entry Time'].dt.year == year]
#     if trades_this_year.empty:
#         print(f"  No trades for {year}")
#         continue

#     # Load options DataFrames for this year (make sure they're in your environment)
#     options_df = globals().get(f"options_df_{year}")
#     options_df_2 = globals().get(f"options_df_2_{year}")

#     if options_df is None or options_df_2 is None:
#         print(f"  Option data missing for {year}")
#         continue

#     year_results = []
#     for idx, row in trades_this_year.iterrows():
#         res = find_best_strike_for_signal(row, options_df, options_df_2)
#         year_results.append(res)

#     # Add non-empty results to master list
#     year_results = [x for x in year_results if x is not None]
#     all_results.extend(year_results)

# # Combine into final DataFrame
# strikes_df = pd.DataFrame(all_results)
# print(strikes_df)


In [ ]:
# import glob
# import os
# import pandas as pd
# from collections import defaultdict

# def get_pocket_file(date, kind):
#     all_files = glob.glob(f"./options_data_quarterly/{kind}_*.parquet")
#     for file in all_files:
#         tokens = file.split("_")
#         date1 = pd.to_datetime(tokens[-2])
#         date2 = pd.to_datetime(tokens[-1].replace(".parquet", ""))
#         if date1 <= date <= date2:
#             return file
#     return None

# all_results = []
# # Group by pocket file (pocket_id = pocket_file_base: YYYYMMDD_YYYYMMDD)
# pocket_trades = defaultdict(list)

# for idx, row in trade_log_df.iterrows():
#     entry_dt = row['Entry Time']
#     nearest_file = get_pocket_file(entry_dt, "options_nearest")
#     next_file = get_pocket_file(entry_dt, "options_next")
#     pocket_id = None
#     if nearest_file and next_file:
#         # Use a shared base, e.g. by period
#         pocket_id = nearest_file.split("_")[-2] + "_" + nearest_file.split("_")[-1]
#     elif nearest_file:
#         pocket_id = nearest_file.split("_")[-2] + "_" + nearest_file.split("_")[-1]
#     elif next_file:
#         pocket_id = next_file.split("_")[-2] + "_" + next_file.split("_")[-1]
#     else:
#         print(f"No pocket file for trade at {entry_dt}")
#         continue
#     pocket_trades[(pocket_id, nearest_file, next_file)].append((idx, row))

# for (pocket_id, nearest_file, next_file), trades in pocket_trades.items():
#     print(f"Processing pocket {pocket_id} with {len(trades)} trades")
#     options_df = pd.read_parquet(nearest_file) if nearest_file and os.path.exists(nearest_file) else None
#     options_df_2 = pd.read_parquet(next_file) if next_file and os.path.exists(next_file) else None
#     if options_df is None or options_df_2 is None:
#         print(f"  Skipping pocket {pocket_id} (missing data file)")
#         continue

#     for idx, row in trades:
#         res = find_best_strike_for_signal(row, options_df, options_df_2)
#         if res is not None:
#             all_results.append(res)

# strikes_df = pd.DataFrame(all_results)
# print(strikes_df)


In [ ]:
# strikes_df

In [ ]:
import os
import pandas as pd
from collections import defaultdict


def get_pocket_file(date, kind):
    all_files = glob.glob(f"./options_data_quarterly/{kind}_*.parquet")
    for file in all_files:
        tokens = file.split("_")
        date1 = pd.to_datetime(tokens[-2])
        date2 = pd.to_datetime(tokens[-1].replace(".parquet", ""))
        if date1 <= date <= date2:
            return file
    return None

all_results = []
pocket_trades = defaultdict(list)

# 1. BUILD TRADE GROUPS BY POCKETS, DEBUG IF POCKET IS MISSING
skipped_no_pocket = []
for idx, row in trade_log_df.iterrows():
    entry_dt = row['Entry Time']
    nearest_file = get_pocket_file(entry_dt, "options_nearest")
    next_file = get_pocket_file(entry_dt, "options_next")
    pocket_id = None
    if nearest_file and next_file:
        pocket_id = nearest_file.split("_")[-2] + "_" + nearest_file.split("_")[-1]
    elif nearest_file:
        pocket_id = nearest_file.split("_")[-2] + "_" + nearest_file.split("_")[-1]
    elif next_file:
        pocket_id = next_file.split("_")[-2] + "_" + next_file.split("_")[-1]
    else:
        print(f"[NO POCKET FILE] Trade idx={idx}, entry_dt={entry_dt}")
        skipped_no_pocket.append((idx, row))
        continue
    pocket_trades[(pocket_id, nearest_file, next_file)].append((idx, row))

# 2. PROCESS EACH POCKET, LOG IF DATA IS MISSING
skipped_missing_data = []
skipped_none_result = []

for (pocket_id, nearest_file, next_file), trades in pocket_trades.items():
    print(f"\nProcessing pocket {pocket_id}: {len(trades)} trades")
    options_df = pd.read_parquet(nearest_file) if nearest_file and os.path.exists(nearest_file) else None
    options_df_2 = pd.read_parquet(next_file) if next_file and os.path.exists(next_file) else None
    if options_df is None or options_df_2 is None:
        print(f"  [SKIP POCKET] Missing data file for pocket {pocket_id} | nearest_file={nearest_file} | next_file={next_file}")
        for idx, row in trades:
            skipped_missing_data.append((idx, row))
        continue

    for idx, row in trades:
        try:
            res = find_best_strike_for_signal(row, options_df, options_df_2)
            if res is not None:
                all_results.append(res)
            else:
                print(f"  [NO STRIKE FOUND] idx={idx} | entry_dt={row['Entry Time']}")
                skipped_none_result.append((idx, row))
        except Exception as e:
            print(f"  [EXCEPTION] idx={idx} | entry_dt={row['Entry Time']} | Error: {e}")
            skipped_none_result.append((idx, row))

# 3. SHOW COUNTS AND OPTIONALLY DETAILS
print("\nSUMMARY:")
print(f"Total trades in log: {len(trade_log_df)}")
print(f"  Trades skipped due to missing pocket: {len(skipped_no_pocket)}")
print(f"  Trades skipped due to missing data file: {len(skipped_missing_data)}")
print(f"  Trades skipped due to None result/exception: {len(skipped_none_result)}")
print(f"  Trades with valid strike: {len(all_results)}")

if len(skipped_no_pocket) > 0:
    print("\nSample missing pocket trades:")
    print(pd.DataFrame([dict(row) for _, row in skipped_no_pocket]).head())

if len(skipped_missing_data) > 0:
    print("\nSample missing data file trades:")
    print(pd.DataFrame([dict(row) for _, row in skipped_missing_data]).head())

if len(skipped_none_result) > 0:
    print("\nSample None/exception trades:")
    print(pd.DataFrame([dict(row) for _, row in skipped_none_result]).head())

# 4. Final Results DataFrame
strikes_df = pd.DataFrame(all_results)
print("\nFinal strikes_df shape:", strikes_df.shape)


In [ ]:
import os
import pandas as pd
from collections import defaultdict
import glob

# Gather all pocket files sorted by start date
def get_all_pocket_files(kind):
    all_files = glob.glob(f"./options_data_quarterly/{kind}_*.parquet")
    pockets = []
    for file in all_files:
        tokens = file.split("_")
        date1 = pd.to_datetime(tokens[-2])
        date2 = pd.to_datetime(tokens[-1].replace(".parquet", ""))
        pockets.append((date1, date2, file))
    pockets.sort()
    return pockets

nearest_pockets = get_all_pocket_files("options_nearest")
next_pockets = get_all_pocket_files("options_next")

# Find the pocket covering date and its neighbors
def get_adjacent_pocket_files(date, pockets):
    prev_file = main_file = next_file = None
    for i, (start, end, file) in enumerate(pockets):
        if start <= date <= end:
            main_file = file
            if i > 0:
                prev_file = pockets[i - 1][2]
            if i + 1 < len(pockets):
                next_file = pockets[i + 1][2]
            break
    return main_file, prev_file, next_file

all_results = []
skipped_all_missing = []
skipped_none_result = []

for idx, row in trade_log_df.iterrows():
    entry_dt = row['Entry Time']

    # Find all possible pocket files for this date (nearest & next expiry)
    n_main, n_prev, n_next = get_adjacent_pocket_files(entry_dt, nearest_pockets)
    x_main, x_prev, x_next = get_adjacent_pocket_files(entry_dt, next_pockets)

    # Helper to try all combinations in main, previous, and next pockets
    tried_pockets = []
    found = False
    for nfile in [n_main, n_prev, n_next]:
        for xfile in [x_main, x_prev, x_next]:
            if (nfile is None and xfile is None) or (nfile, xfile) in tried_pockets:
                continue
            tried_pockets.append((nfile, xfile))
            options_df = pd.read_parquet(nfile) if nfile and os.path.exists(nfile) else None
            options_df_2 = pd.read_parquet(xfile) if xfile and os.path.exists(xfile) else None
            if options_df is None and options_df_2 is None:
                continue
            try:
                res = find_best_strike_for_signal(row, options_df, options_df_2)
                if res is not None:
                    all_results.append(res)
                    found = True
                    break
            except Exception as e:
                print(f"[EXCEPTION] idx={idx} | entry_dt={row['Entry Time']} | Error: {e}")
                skipped_none_result.append((idx, row))
        if found:
            break
    if not found:
        skipped_all_missing.append((idx, row))

# Print summary
print("\nSUMMARY:")
print(f"Total trades in log: {len(trade_log_df)}")
print(f"  Trades skipped after all pockets tried: {len(skipped_all_missing)}")
print(f"  Trades skipped due to None/exception: {len(skipped_none_result)}")
print(f"  Trades with valid strike: {len(all_results)}")

if skipped_all_missing:
    print("\nSample trades missing after all pockets tried:")
    print(pd.DataFrame([dict(row) for _, row in skipped_all_missing]).head())

if skipped_none_result:
    print("\nSample trades with None/exception:")
    print(pd.DataFrame([dict(row) for _, row in skipped_none_result]).head())

strikes_df = pd.DataFrame(all_results)
print("\nFinal strikes_df shape:", strikes_df.shape)


In [ ]:
strikes_df

In [ ]:
strikes_df

In [ ]:
#sorting strikes by time
# If Entry Time is not already a datetime type:
strikes_df['Entry Time'] = pd.to_datetime(strikes_df['Entry Time'])
trade_log_df['Entry Time'] = pd.to_datetime(trade_log_df['Entry Time'])

# Sort by Entry Time
trade_log_df_sorted = trade_log_df.sort_values(by='Entry Time').reset_index(drop=True)

# (Optional) Overwrite the original
# strikes_df = strikes_df_sorted

print(trade_log_df_sorted.tail())

# Sort by Entry Time
strikes_df_sorted = strikes_df.sort_values(by='Entry Time').reset_index(drop=True)

# (Optional) Overwrite the original
# strikes_df = strikes_df_sorted

print(strikes_df_sorted.tail())


In [ ]:
#merge strikes and trade logs
# Ensure both are using datetime for Entry Time
strikes_df_sorted['Entry Time'] = pd.to_datetime(strikes_df_sorted['Entry Time'])
trade_log_df['Entry Time'] = pd.to_datetime(trade_log_df['Entry Time'])

# Merge on Entry Time (keeps strikes_df_sorted order)
strikes_df_merged = strikes_df_sorted.merge(
    trade_log_df[['Entry Time', 'Exit Time']],
    on='Entry Time',
    how='left'
)

# Optional: move Exit Time next to Entry Time for readability
cols = list(strikes_df_merged.columns)
# Move Exit Time just after Entry Time
exit_idx = cols.index('Entry Time') + 1
cols.insert(exit_idx, cols.pop(cols.index('Exit Time')))
strikes_df_merged = strikes_df_merged[cols]

print(strikes_df_merged.head())


In [ ]:
#final exit using merged options_df
def get_exit_option_price_from_merged(row, options_df_merged):

    entry_time = pd.to_datetime(row['Entry Time'])
    exit_time = pd.to_datetime(row['Exit Time'])
    strike = float(row['Strike'])
    expiry = pd.to_datetime(row['Expiry'])
    option_type = row['Option Type']  # 'PE' or 'CE'

    # Ensure correct dtypes for filtering
    if not pd.api.types.is_datetime64_any_dtype(options_df_merged['ExpiryDate']):
        options_df_merged['ExpiryDate'] = pd.to_datetime(options_df_merged['ExpiryDate'], errors='coerce')
    if not pd.api.types.is_datetime64_any_dtype(options_df_merged['DateTime']):
        options_df_merged['DateTime'] = pd.to_datetime(options_df_merged['DateTime'], errors='coerce')

    expiry_date = expiry.normalize()
    expiry_1515 = expiry_date + pd.Timedelta(hours=15, minutes=15)

    print(f"[DEBUG] Entry {entry_time}, Exit {exit_time}, Strike {strike}, Expiry {expiry_date.date()}, Type {option_type}")

    # 1. If exited AFTER expiry day 15:15, force exit at expiry day 15:15
    if exit_time > expiry_1515:
        # Find all ticks for this expiry, strike, and type ON expiry day <= 15:15
        mask = (
            (options_df_merged['ExpiryDate'] == expiry_date) &
            (np.isclose(options_df_merged['StrikePrice'].astype(float), strike, atol=0.01)) &
            (options_df_merged['Type'] == option_type) &
            (options_df_merged['DateTime'].dt.date == expiry_date.date()) &
            (options_df_merged['DateTime'].dt.time <= pd.to_datetime('15:15').time())
        )
        expiry_opts = options_df_merged[mask].copy()
        if not expiry_opts.empty:
            last_tick = expiry_opts.sort_values('DateTime').iloc[-1]
            return last_tick['Open']
        else:
            print(f"[DEBUG] No option found after expiry for Entry Time {entry_time}, Expiry {expiry_date}, Strike {strike}, Type {option_type}")
            return np.nan

    # 2. Exited ON expiry day, but BEFORE or AT 15:15, or on any day before expiry
    else:
        mask = (
            (options_df_merged['ExpiryDate'] == expiry_date) &
            (np.isclose(options_df_merged['StrikePrice'].astype(float), strike, atol=0.01)) &
            (options_df_merged['Type'] == option_type) &
            (options_df_merged['DateTime'].dt.date == exit_time.date())
        )
        opts = options_df_merged[mask].copy()
        if opts.empty:
            print(f"[DEBUG] No option found for exit for Entry {entry_time}, Exit {exit_time}, Expiry {expiry_date}, Strike {strike}, Type {option_type}")
            return np.nan
        time_diffs = abs(opts['DateTime'] - exit_time)
        min_idx = time_diffs.idxmin()
        if time_diffs[min_idx] <= pd.Timedelta(minutes=0):
            return opts.loc[min_idx, 'Open']
        else:
            print(f"[DEBUG] No tick within ±2min for Entry {entry_time} (exit {exit_time}), got diff {time_diffs[min_idx]}")
            return np.nan


In [ ]:
# exit_prices_dict = {}

# for year in sorted(strikes_df_merged['Exit Time'].dt.year.unique()):
#     print(f"Processing Exit Option Price for year: {year}")
#     year_strikes = strikes_df_merged[strikes_df_merged['Exit Time'].dt.year == year]
#     options_df_merged = globals().get(f"options_df_merged_{year}")
#     if options_df_merged is None:
#         print(f"  options_df_merged_{year} not found! Skipping.")
#         continue

#     for idx, row in year_strikes.iterrows():
#         price = get_exit_option_price_from_merged(row, options_df_merged)
#         exit_prices_dict[idx] = price

# # Reassign by mapping index to correct price
# strikes_df_merged['Exit Option Price'] = strikes_df_merged.index.map(exit_prices_dict)


In [ ]:
#LOOPING THROUGH 3 MONTH POCKETS


# Gather all merged pockets
merged_pockets = []
all_files = glob.glob("./options_data_quarterly/options_merged_*.parquet")
for file in all_files:
    tokens = file.split("_")
    date1 = pd.to_datetime(tokens[-2])
    date2 = pd.to_datetime(tokens[-1].replace(".parquet", ""))
    merged_pockets.append((date1, date2, file))
merged_pockets.sort()

def get_adjacent_merged_pocket_files(date, pockets=merged_pockets):
    prev_file = main_file = next_file = None
    for i, (start, end, file) in enumerate(pockets):
        if start <= date <= end:
            main_file = file
            if i > 0:
                prev_file = pockets[i - 1][2]
            if i + 1 < len(pockets):
                next_file = pockets[i + 1][2]
            break
    return main_file, prev_file, next_file

exit_prices_dict = {}

for idx, row in strikes_df_merged.iterrows():
    exit_dt = row['Exit Time']
    found = False

    # Try main, prev, and next pockets (fallback logic)
    main_file, prev_file, next_file = get_adjacent_merged_pocket_files(exit_dt)
    tried = []
    for pocket_file in [main_file, prev_file, next_file]:
        if pocket_file is None or pocket_file in tried:
            continue
        tried.append(pocket_file)
        options_df_merged = pd.read_parquet(pocket_file)
        price = get_exit_option_price_from_merged(row, options_df_merged)
        if price is not None:
            exit_prices_dict[idx] = price
            found = True
            break
    if not found:
        print(f"[NO EXIT PRICE] idx={idx}, Exit Time={exit_dt}")

# Assign by mapping
strikes_df_merged['Exit Option Price'] = strikes_df_merged.index.map(exit_prices_dict)


In [ ]:
# Show first few fetched exit prices
print(strikes_df_merged.tail(20))

# Show only rows where fetching exit price failed
print("\nRows with missing exit price:")
print(strikes_df_merged[strikes_df_merged['Exit Option Price'].isna()])


TRYING TO USE DASK

In [ ]:
# Combine all yearwise merged DataFrames into one
options_df_merged_all = pd.concat(merged_dict.values(), ignore_index=True)

# (Optional) Drop duplicates across all years just in case
options_df_merged_all = options_df_merged_all.drop_duplicates(
    subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
).reset_index(drop=True)

# (Optional) Sort for easier use
options_df_merged_all = options_df_merged_all.sort_values(
    ['DateTime', 'ExpiryDate', 'StrikePrice', 'Type']
).reset_index(drop=True)

# Save if needed
options_df_merged_all_path = os.path.join(data_dir, "options_df_merged_all.parquet")
options_df_merged_all.to_parquet(options_df_merged_all_path, index=False)
print(f"✅ Saved ALL merged options data to {options_df_merged_all_path}")



In [ ]:
strikes_df

In [ ]:
#SEPARATE FUNCTION TO CHANGE EXIT TIME IF EXPIRED

def adjust_exit_times_after_expiry(strikes_df):
    # Ensure columns are datetime
    strikes_df['Exit Time'] = pd.to_datetime(strikes_df['Exit Time'])
    strikes_df['Expiry'] = pd.to_datetime(strikes_df['Expiry'])
    
    adjusted_exit_times = []
    for idx, row in strikes_df.iterrows():
        expiry_dt = row['Expiry'].replace(hour=15, minute=15, second=0, microsecond=0)
        # If Exit Time is after expiry 15:15, set to expiry 15:15
        if row['Exit Time'] > expiry_dt:
            adjusted_exit_times.append(expiry_dt)
        else:
            adjusted_exit_times.append(row['Exit Time'])
    strikes_df['Exit Time'] = adjusted_exit_times
    return strikes_df

# Usage
strikes_df_merged = adjust_exit_times_after_expiry(strikes_df_merged)


In [ ]:
strikes_df_merged

In [ ]:


# Convert DataFrames to Dask
strikes_dd = dd.from_pandas(strikes_df_merged, npartitions=4)
options_dd = dd.from_pandas(options_df_merged_all, npartitions=8)

# Normalize types
strikes_dd['Strike'] = strikes_dd['Strike'].astype(float)
strikes_dd['Expiry'] = dd.to_datetime(strikes_dd['Expiry'])
strikes_dd['Option Type'] = strikes_dd['Option Type'].str.upper()
strikes_dd['Exit Time'] = dd.to_datetime(strikes_dd['Exit Time'])

options_dd['StrikePrice'] = options_dd['StrikePrice'].astype(float)
options_dd['ExpiryDate'] = dd.to_datetime(options_dd['ExpiryDate'])
options_dd['Type'] = options_dd['Type'].str.upper()
options_dd['DateTime'] = dd.to_datetime(options_dd['DateTime'])

# Create rounded minute for fuzzy merge (for ±1min, create both floor and ceil if needed)
strikes_dd['ExitTime_Round'] = strikes_dd['Exit Time'].dt.floor('min')
options_dd['DateTime_Round'] = options_dd['DateTime'].dt.floor('min')

# Merge on ExitTime_Round, Expiry, Strike, Option Type
merged = dd.merge(
    strikes_dd,
    options_dd,
    left_on=['ExitTime_Round', 'Expiry', 'Strike', 'Option Type'],
    right_on=['DateTime_Round', 'ExpiryDate', 'StrikePrice', 'Type'],
    suffixes=('', '_opt'),
    how='left'
)

# Now, 'Open' column from options_dd is the exit price.
# Optional: If multiple matches per merge key, take the nearest DateTime to Exit Time

def pick_nearest_exit(row):
    # If there are duplicates, keep the row where DateTime is closest to Exit Time
    if pd.isna(row['DateTime']):
        return np.nan
    time_diff = abs((row['Exit Time'] - row['DateTime']).total_seconds())
    if time_diff <= 60:
        return row['Open']
    else:
        return np.nan

merged['Exit Option Price'] = merged.apply(pick_nearest_exit, axis=1, meta=('Exit Option Price', 'float64'))

# Select only original columns plus Exit Option Price
result_cols = list(strikes_df_merged.columns) + ['Exit Option Price']
result_dd = merged[result_cols]

# Compute the final DataFrame
strikes_with_exit_prices = result_dd.compute()

print(strikes_with_exit_prices.head())


In [ ]:
# #vatsalya code for melt


# # === Step 1: Load your data ===
# # If data is already loaded, skip this part
# # straddle_df = pd.read_csv("straddle_input.csv")
# # df_options = pd.read_csv("options_data.csv")

# # === Step 2: Filter straddle_df to only matching dates ===
# # Ensure consistent date format
# df_options['Date'] = df_options['Date'].astype(str).str.strip()
# valid_dates = df_options['Date'].unique()

# straddle_df['Date'] = straddle_df['Date'].astype(str).str.strip()
# straddle_df = straddle_df[straddle_df['Date'].isin(valid_dates)]

# # === Step 3: Prepare strike columns from -1000 to +1000 in 100 steps ===
# offsets = list(range(-500, 501, 50))  # [-1000, -900, ..., 0, ..., +1000]
# strike_columns = ['ATM_Strike'] + [f'Strike_{offset:+}' for offset in offsets]

# # Ensure all strike columns exist (in case some are missing)
# existing_strike_columns = [col for col in strike_columns if col in straddle_df.columns]
# straddle_df[existing_strike_columns] = straddle_df[existing_strike_columns].astype('string[python]')

# # Melt the DataFrame to long format for strike matching
# melted = straddle_df[['Date', 'Time'] + existing_strike_columns].melt(
#     id_vars=['Date', 'Time'],
#     var_name='Strangle_Type',
#     value_name='StrikePrice'
# ).dropna(subset=['StrikePrice'])

# # Format fields
# melted['Date'] = melted['Date'].astype(str).str.strip()
# melted['Time'] = melted['Time'].astype(str).str.strip().str[:5]  # Truncate to HH:MM
# melted['StrikePrice'] = melted['StrikePrice'].astype(float).astype(int)

# print(f"🔄 Melted straddle_df to {len(melted)} rows for matching...")

# # === Step 4: Clean df_options ===
# df_options['Time'] = df_options['Time'].astype(str).str.strip().str[:5]
# df_options['StrikePrice'] = df_options['StrikePrice'].astype(float).astype(int)

# # Convert to Dask DataFrame
# df_options_dd = dd.from_pandas(df_options, npartitions=8)

# # === Step 5: Perform merge with Dask ===
# print("🔍 Performing fast vectorized join using Dask...")

# merged_dd = dd.merge(
#     melted,
#     df_options_dd[['Date', 'Time', 'StrikePrice', 'ExpiryDate', 'Open','Type', 'Ticker']],
#     on=['Date', 'Time', 'StrikePrice'],
#     how='inner'
# )

# # === Step 6: Finalize and Save ===
# final_df = merged_dd.compute()

# print(f"✅ Total matched rows: {len(final_df)}")
# print(final_df.head())

# # Save result
# final_df.to_csv("NIFTY_strangle_filtered_options_rows.csv", index=False)
# print("💾 Saved to 'SENSEX_strangle_filtered_options_rows.csv'")


RUN OVER MISSING DATA

In [ ]:
import glob
import pandas as pd
import os

data_dir = "./options_data_quarterly"

# 1. Gather all merged pocket files
merged_files = sorted(glob.glob(os.path.join(data_dir, "options_merged_*.parquet")))

# 2. Load and concatenate all pockets
dfs = []
for file in merged_files:
    print(f"Loading {file} ...")
    df = pd.read_parquet(file)
    dfs.append(df)

options_df_merged_all = pd.concat(dfs, ignore_index=True)

# 3. Drop duplicates across all periods
options_df_merged_all = options_df_merged_all.drop_duplicates(
    subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
).reset_index(drop=True)

# 4. Sort for usability
options_df_merged_all = options_df_merged_all.sort_values(
    ['DateTime', 'ExpiryDate', 'StrikePrice', 'Type']
).reset_index(drop=True)

# 5. Save as master file
all_path = os.path.join(data_dir, "options_df_merged_all.parquet")
options_df_merged_all.to_parquet(all_path, index=False)
print(f"✅ Saved ALL merged options data to {all_path}")


In [ ]:
#RUN THE MISSING EXIT PRICE VALUES OVER MERGED OPTIONS DATA 
# 1. Identify missing Exit Option Price rows
missing_exit_mask = strikes_df_merged['Exit Option Price'].isna()

# 2. Run get_exit_option_price_from_merged only for those rows
for idx in strikes_df_merged[missing_exit_mask].index:
    row = strikes_df_merged.loc[idx]
    exit_price = get_exit_option_price_from_merged(row, options_df_merged_all)
    # If you want to know which ones remain missing, print them
    if exit_price is None or pd.isna(exit_price):
        print(f"Still missing for idx {idx}: {row.to_dict()}")
    # 3. Assign found value back
    strikes_df_merged.at[idx, 'Exit Option Price'] = exit_price

# Now strikes_df_merged should have more complete exit prices!



In [ ]:
strikes_df_merged

In [ ]:
strikes_df_merged

In [ ]:
# Show only rows where fetching exit price failed
print("\nRows with missing exit price:")
print(strikes_df_merged[strikes_df_merged['Exit Option Price'].isna()])

In [ ]:
trades_df=strikes_df_merged

In [ ]:
trades_df = trades_df.rename(columns={"Option Price": "Entry Price"})
trades_df = trades_df.rename(columns={"Exit Option Price": "Exit Price"})
trades_df.to_excel("trades_df.xlsx", index=False)
trades_df

In [ ]:
#TRANSACTION COSTS
# Lot sizes for reference (already in your code)
lot_sizes = {"NIFTY": 75, "BANKNIFTY": 30, "FINNIFTY": 40}
lot_multiplier = 1  # Use as per your actual qty scaling

# Spread
def get_spread(value):
    if 0 <= value <= 10:
        spread_percentage = 0.05
    else:
        spread_percentage = 0.1
    return (value * spread_percentage) / 100

# Charges calculation (as per your logic)
def calculate_option_trade_charges(buy_price, sell_price, lot_size, brokerage_per_order=0):
    etc_rate = 0.00035
    sebi_rate = 0.000001
    stamp_duty_rate = 0.00003
    stt_rate = 0.001
    gst_rate = 0.18

    turnover = (buy_price + sell_price) * lot_size
    etc = etc_rate * turnover
    sebi = sebi_rate * turnover
    stamp_duty = stamp_duty_rate * (buy_price * lot_size)
    stt = stt_rate * (sell_price * lot_size)
    brokerage = 2 * brokerage_per_order
    gst = gst_rate * (etc + sebi + brokerage)
    total_charges = etc + sebi + stamp_duty + stt + brokerage + gst
    gross_pnl = (sell_price - buy_price) * lot_size
    net_pnl = gross_pnl - total_charges

    return {
        'Turnover': turnover,
        'ETC': etc,
        'SEBI': sebi,
        'Stamp Duty': stamp_duty,
        'STT': stt,
        'Brokerage': brokerage,
        'GST': gst,
        'Total Charges': total_charges,
        'Gross P&L': gross_pnl,
        'Net P&L': net_pnl
    }


In [ ]:

#trades_df = pd.read_excel("trade_log.xlsx") # If loading from csv
# Example: Option Lot Size
index_type = "NIFTY"  # or fetch as needed
lot_size = lot_sizes[index_type]
final_lot_size = lot_size * lot_multiplier  # if you are scaling


# Apply spread
trades_df["Entry_Spread"] = trades_df["Entry Price"].apply(get_spread)
trades_df["Exit_Spread"] = trades_df["Exit Price"].apply(get_spread)

# Slippage logic: for shorting, you SELL first and BUY to cover
# Assume for shorts: you sell at (Entry Price - Spread), buy to cover at (Exit Price + Spread)
trades_df["Adj_Entry_Price"] = (trades_df["Entry Price"] - trades_df["Entry_Spread"]).round(2)
trades_df["Adj_Exit_Price"] = (trades_df["Exit Price"] + trades_df["Exit_Spread"]).round(2)

# Now calculate Net PnL with charges
def compute_net_leg_pnl(entry, exit, lot):
    # For shorts: Entry is sell, Exit is buy (we use negative quantity in some cases, keep consistent)
    # For your logic: always (SELL at Entry, BUY at Exit) for short
    return calculate_option_trade_charges(buy_price=exit, sell_price=entry, lot_size=lot)["Net P&L"]

trades_df["Net_PnL_With_Spread"] = trades_df.apply(
    lambda row: compute_net_leg_pnl(row["Adj_Entry_Price"], row["Adj_Exit_Price"], final_lot_size),
    axis=1
)

# If you want also PnL without spread/costs:
def compute_gross_pnl(entry, exit, lot):
    return (entry - exit) * lot

trades_df["Gross_PnL"] = trades_df.apply(
    lambda row: compute_gross_pnl(row["Entry Price"], row["Exit Price"], final_lot_size),
    axis=1
)


In [ ]:
trades_df['Entry Time'] = pd.to_datetime(trades_df['Entry Time'])
trades_df['Exit Time'] = pd.to_datetime(trades_df['Exit Time'])
trades_df['Year'] = trades_df['Exit Time'].dt.year
trades_df['Date'] = trades_df['Exit Time'].dt.date
trades_df = trades_df.sort_values('Exit Time').reset_index(drop=True)

# --- Helper Functions ---
def calculate_drawdown(pnl_series):
    cumulative = pnl_series.cumsum()
    high_watermark = cumulative.cummax()
    drawdown = cumulative - high_watermark
    max_dd = drawdown.min()
    max_dd_pct = max_dd / high_watermark.max() if high_watermark.max() != 0 else 0
    return max_dd, max_dd_pct

def sortino_ratio(returns, periods):
    mean_return = np.mean(returns)
    downside = returns[returns < 0]
    downside_deviation = np.std(downside, ddof=1) if len(downside) > 0 else 1e-9
    return mean_return / downside_deviation * np.sqrt(periods)

# --- Define Periods ---
last_date = trades_df['Exit Time'].max()
years = sorted(trades_df['Year'].unique())
periods = {f"Year {y}": [pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")] for y in years}
periods.update({
    'Last 6 Months': [last_date - pd.DateOffset(months=6), last_date],
    'Last 1 Year':   [last_date - pd.DateOffset(years=1), last_date],
    'Last 2 Years':  [last_date - pd.DateOffset(years=2), last_date],
    'Full Period':   [trades_df['Exit Time'].min(), trades_df['Exit Time'].max()]
})

summary_rows = []

for period_name, (start_time, end_time) in periods.items():
    # Filter data for period
    chunk = trades_df[
        (trades_df['Exit Time'] >= start_time) & (trades_df['Exit Time'] < end_time)
    ].copy()
    if chunk.empty:
        continue

    chunk['Date'] = chunk['Exit Time'].dt.date
    daily_df = chunk.groupby('Date').agg(
        Net_PnL_With_Spread=('Net_PnL_With_Spread', 'sum'),
        Gross_PnL=('Gross_PnL', 'sum')
    ).reset_index()
    daily_df['Daily_Return'] = daily_df['Net_PnL_With_Spread'] / daily_df['Net_PnL_With_Spread'].abs().sum()

    # Overall stats
    returns = daily_df['Daily_Return'].fillna(0)
    win_rate = (chunk['Net_PnL_With_Spread'] > 0).mean()
    max_dd, max_dd_pct = calculate_drawdown(daily_df['Net_PnL_With_Spread'])
    sortino_12 = sortino_ratio(returns, 12)
    sortino_48 = sortino_ratio(returns, 48)
    total_return = daily_df['Net_PnL_With_Spread'].sum()
    annualized_return = (1 + returns.mean()) ** 252 - 1 if len(returns) > 0 else np.nan
    z_score = (returns - returns.mean()) / returns.std(ddof=0) if returns.std(ddof=0) > 0 else np.nan

    # Bullish
    chunk_bull = chunk[chunk['Side'] == 'Bullish']
    bullish_pnl = chunk_bull['Net_PnL_With_Spread'].sum()
    bullish_win = (chunk_bull['Net_PnL_With_Spread'] > 0).mean() if len(chunk_bull) > 0 else np.nan
    bull_dd, bull_dd_pct = calculate_drawdown(
        chunk_bull.groupby('Date')['Net_PnL_With_Spread'].sum() if not chunk_bull.empty else pd.Series([0])
    )

    # Bearish
    chunk_bear = chunk[chunk['Side'] == 'Bearish']
    bearish_pnl = chunk_bear['Net_PnL_With_Spread'].sum()
    bearish_win = (chunk_bear['Net_PnL_With_Spread'] > 0).mean() if len(chunk_bear) > 0 else np.nan
    bear_dd, bear_dd_pct = calculate_drawdown(
        chunk_bear.groupby('Date')['Net_PnL_With_Spread'].sum() if not chunk_bear.empty else pd.Series([0])
    )

    summary_rows.append({
        "Period": period_name,
        "Total Net PnL (With Spread)": total_return,
        "Total Gross PnL": daily_df['Gross_PnL'].sum(),
        "Bullish PnL": bullish_pnl,
        "Bullish Win %": round(bullish_win*100,2) if pd.notnull(bullish_win) else np.nan,
        "Bullish Max DD": bull_dd,
        "Bullish Max DD %": bull_dd_pct * 100,
        "Bearish PnL": bearish_pnl,
        "Bearish Win %": round(bearish_win*100,2) if pd.notnull(bearish_win) else np.nan,
        "Bearish Max DD": bear_dd,
        "Bearish Max DD %": bear_dd_pct * 100,
        "Win Rate": round(win_rate*100,2),
        "Sortino (12)": sortino_12,
        "Sortino (48)": sortino_48,
        "Max Drawdown": max_dd,
        "Max Drawdown %": max_dd_pct * 100,
        "Annualized Return %": round(annualized_return*100,2) if not pd.isnull(annualized_return) else np.nan,
        "Mean Daily Return": returns.mean(),
        "Std Dev Daily Return": returns.std(),
        "Z-Score Mean": z_score.mean() if isinstance(z_score, pd.Series) else np.nan,
        "Z-Score Std": z_score.std() if isinstance(z_score, pd.Series) else np.nan
    })

summary_df = pd.DataFrame(summary_rows)

# --- Write to Excel ---
with pd.ExcelWriter("analytics_report.xlsx", engine='xlsxwriter') as writer:
    # Sheet 1: All trades (full trade log, easy for Excel filter/sort)
    trades_df.to_excel(writer, index=False, sheet_name="All_Trades")
    # Sheet 2: Analytics summary
    summary_df.to_excel(writer, index=False, sheet_name="Analytics_Summary")

print("✅ Clean analytics Excel written: analytics_report.xlsx")